# Rossmann Store Sales: Per-Store LightGBM Forecasting

This notebook trains a store-level LightGBM model on individual store-day rows (vs. the chain-wide aggregation in notebook 03). 
The trained model powers the `POST /forecast/store` and `POST /forecast/store/whatif` API endpoints.

**Key differences from notebook 03 (chain-wide model):**
- Data: individual store-day rows (not daily aggregates across all 1,115 stores)
- Features: includes `Store` ID + store metadata from `store.csv` (StoreType, Assortment, CompetitionDistance, Promo2)
- Lag features: computed per-store using `groupby('Store').shift()`
- Target: individual store daily sales (range ~0–40,000)
- Dropped: `DayOfWeek_Num` (had zero importance in chain model)

**Saved artifacts:**
- `models/lightgbm_store_model.pkl` — trained LGBMRegressor
- `models/store_features.pkl` — encoded store metadata dict keyed by Store int

## 1. Imports

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

ROOT = Path("..").resolve()
TRAIN_CSV = ROOT / "Data" / "raw" / "train.csv"
STORE_CSV = ROOT / "Data" / "raw" / "store.csv"
MODEL_OUT = ROOT / "models" / "lightgbm_store_model.pkl"
FEATURES_OUT = ROOT / "models" / "store_features.pkl"

SPLIT_DATE = "2015-06-01"

FEATURES = [
    "Store",
    "DayOfWeek",
    "Month",
    "Year",
    "DayOfMonth",
    "Promo",
    "SchoolHoliday",
    "StateHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Promo2",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
]

## 2. Data Loading

In [ ]:
train = pd.read_csv(TRAIN_CSV, low_memory=False)
store = pd.read_csv(STORE_CSV)
print(f"train shape : {train.shape}")
print(f"store shape : {store.shape}")
train.head(3)

## 3. Encode Store Metadata

In [ ]:
def encode_store_meta(store_df: pd.DataFrame) -> pd.DataFrame:
    """Label-encode categorical store columns and fill missing values."""
    s = store_df.copy()
    type_map = {"a": 1, "b": 2, "c": 3, "d": 4}
    assort_map = {"a": 1, "b": 2, "c": 3}
    s["StoreType"] = s["StoreType"].str.lower().map(type_map).fillna(0).astype(int)
    s["Assortment"] = s["Assortment"].str.lower().map(assort_map).fillna(0).astype(int)
    median_dist = s["CompetitionDistance"].median()
    s["CompetitionDistance"] = s["CompetitionDistance"].fillna(median_dist)
    return s[["Store", "StoreType", "Assortment", "CompetitionDistance", "Promo2"]].set_index("Store")

store_meta_idx = encode_store_meta(store)

# Save encoded lookup for the API
store_features_dict = store_meta_idx.to_dict(orient="index")
joblib.dump(store_features_dict, FEATURES_OUT)
print(f"Store features saved → {FEATURES_OUT}")
store_meta_idx.head()

## 4. Feature Engineering

In [ ]:
df = train.copy()
df["Date"] = pd.to_datetime(df["Date"])

# Keep only open store-days (closed days have Sales=0 which corrupts lag features)
df = df[df["Open"] == 1].copy()
print(f"Rows after filtering closed days: {len(df):,}")

# Encode StateHoliday as binary
df["StateHoliday"] = df["StateHoliday"].apply(lambda x: 0 if (x == "0" or x == 0) else 1).astype(int)

# Calendar features
df["DayOfWeek"] = df["Date"].dt.dayofweek + 1   # 1=Mon … 7=Sun
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year
df["DayOfMonth"] = df["Date"].dt.day

# Join store metadata
df = df.join(store_meta_idx, on="Store")

# Sort for grouped shift to work correctly
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

# Per-store lag and rolling features
grp = df.groupby("Store")["Sales"]
df["Sales_Lag_1"] = grp.shift(1)
df["Sales_Lag_7"] = grp.shift(7)
df["Sales_Lag_14"] = grp.shift(14)
df["Sales_Rolling_Mean_7"] = grp.shift(1).groupby(df["Store"]).transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
df["Sales_Rolling_Mean_14"] = grp.shift(1).groupby(df["Store"]).transform(
    lambda x: x.rolling(14, min_periods=1).mean()
)

# Drop warmup rows
df = df.dropna(subset=["Sales_Lag_14"]).reset_index(drop=True)
print(f"Rows after dropping lag warmup NaNs: {len(df):,}")
df[FEATURES + ["Sales"]].head(3)

## 5. Train / Test Split

In [ ]:
split = pd.Timestamp(SPLIT_DATE)
train_df = df[df["Date"] < split]
test_df  = df[df["Date"] >= split]
print(f"Train: {len(train_df):,} rows  ({train_df['Date'].min().date()} → {train_df['Date'].max().date()})")
print(f"Test : {len(test_df):,}  rows  ({test_df['Date'].min().date()} → {test_df['Date'].max().date()})")

X_train, y_train = train_df[FEATURES], train_df["Sales"]
X_test,  y_test  = test_df[FEATURES],  test_df["Sales"]

## 6. Model Training

In [ ]:
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
model.fit(X_train, y_train)
print("Training complete.")

## 7. Evaluation

In [ ]:
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
mask = y_test > 0
rmspe = np.sqrt(np.mean(((y_test[mask] - preds[mask]) / y_test[mask]) ** 2))

print(f"Test MAE  : {mae:,.0f}")
print(f"Test RMSPE: {rmspe:.4f}  ({rmspe*100:.2f}%)")

## 8. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Feature Importance — Per-Store LightGBM", fontsize=14)
ax.set_ylabel("Importance (splits)")
ax.set_xlabel("Feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(ROOT / "figures" / "03_forecasting_modern" / "store_feature_importance.png", dpi=150)
plt.show()
print(importance.to_string())

## 9. Save Model

In [ ]:
joblib.dump(model, MODEL_OUT)
print(f"Model saved → {MODEL_OUT}")
print(f"Store features already saved → {FEATURES_OUT}")